
---

# Inteligencia Artificial y Modelado Predictivo

Diseñamos, entrenamos y validamos una arquitectura neuronal híbrida: una **Red de Memoria a Corto y Largo Plazo (LSTM Dual-Head)** que aprende la microestructura del mercado para eliminar el desfase temporal, previendo el **Residuo de Microestructura** y calculando simultáneamente la varianza estadística (riesgo) de la predicción [2].

---

## 1. Importación AI y División del Master (Celda 1 )
Preparamos el entorno mediante la carga de TensorFlow, Keras y scikit-learn. Recuperamos la fecha exacta del inicio del *Forward Test* para calcular la frontera temporal de los tensores, aislando perfectamente la semana ciega de prueba del conjunto de datos de entrenamiento para prevenir de forma estricta cualquier fuga de información (*data leakage*).

---

## 2. Auditoría del Aprendizaje Residual Histórico (Celda 2 y 3)
Fase de validación científica interna y control de calidad. Se entrena una LSTM sobre una partición del pasado (85% Entrenamiento / 15% Prueba). Generamos un **Dashboard interactivo de 3 paneles** para auditar de forma sincronizada: el seguimiento de la tendencia del filtro, las variaciones dinámicas de la ganancia $K_t$ y el ajuste del LSTM frente a fluctuaciones de ruido histórico no vistas.

---

## 3. Pipeline de Producción y Forward Testing (Celda 4 y 5)
Entrenamos el cerebro predictivo definitivo utilizando el **100% de la data histórica**. Ejecutamos la predicción sobre la semana actual (completamente desconocida para el modelo) y proyectamos el Consenso Central de precio junto con el Canal de Probabilidad Gaussiano ($\pm 2\sigma$) basado en la desviación estándar estimada [2]. Finalmente, serializamos la red (.keras), exportamos los escaladores (.pkl) y generamos la hoja de pronósticos operativa en disco [1, 2].

---

## Resumen de Modelos y Artefactos de Exportación

| Nombre del Archivo | Tipo de Archivo | Propósito en el Pipeline | Estado / Parámetros |
| :--- | :--- | :--- | :--- |
| **`modelo_dual_EURJPY.keras`** | Modelo TensorFlow (Keras) | Red neuronal unificada para predecir residuos y desviación estándar simultáneamente. | Listo para Producción |
| **`scaler_X_dual_EURJPY.pkl`** | Escalador `StandardScaler` | Normalización de características multivariables de entrada en tiempo de ejecución. | Listo para Producción |
| **`scaler_y_res_dual_EURJPY.pkl`** | Escalador `StandardScaler` | Normalización de la microestructura residual (ruido) calculada. | Listo para Producción |
| **`scaler_y_std_dual_EURJPY.pkl`** | Escalador `MinMaxScaler` | Normalización de la varianza estadística predictiva (riesgo). | Listo para Producción |

In [1]:
# =====================================================================
# 1. DEPENDENCIAS AI Y LECTURA DEL MASTER DATASET
# =====================================================================

# --- Librerías de datos ---
import numpy as np                            # Álgebra lineal
import pandas as pd                           # Manipulación de tablas
import joblib                                 # Guardar escaladores (.pkl)

# --- Inteligencia Artificial ---
import tensorflow as tf                       # Redes neuronales
import plotly.graph_objects as go             # Gráficos interactivos
from plotly.subplots import make_subplots     # Subplots dinámicos

# --- Preprocesamiento y Modelado ---
from sklearn.preprocessing import StandardScaler, MinMaxScaler  # Escaladores
from tensorflow.keras.models import Model                       # API Funcional
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout # Capas de red
from tensorflow.keras.callbacks import EarlyStopping            # Control de overfitting
from tensorflow.keras.optimizers import Adam                    # Optimizador

# =====================================================================
# DEFINICIÓN DE VARIABLES
# =====================================================================

symb = "EURJPY"                               # Ticker del activo

# Carga de datos
df_master = pd.read_csv('EURJPY_H1_Master_Features.csv') # Carga dataset maestro
df_master['time'] = pd.to_datetime(df_master['time'])     # Formato de fecha

T = 24                                        # Ventana temporal (Memoria)

# Separación de frontera
df_test_ref = pd.read_csv("EURJPY_H1_Semana_Actual_Prueba.csv") # Dataset de referencia
fecha_inicio_forward = pd.to_datetime(df_test_ref['time'].iloc[0]) # Fecha inicio de test
idx_fwd = df_master[df_master['time'] >= fecha_inicio_forward].index[0] # Índice de corte

print(f"-> Entorno AI listo. Dataset cargado. Frontera Forward en índice: {idx_fwd}")

I0000 00:00:1781843882.720220   35927 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781843883.004379   35927 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781843885.019815   35927 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


-> Entorno AI listo. Dataset cargado. Frontera Forward en índice: 6007


In [2]:
# =====================================================================
# 2. AUDITORÍA HISTÓRICA DE 3 PANELES (VALIDACIÓN INTERNA 100% RESTAURADA)
# =====================================================================
print("-> Generando Auditoría Cuantitativa e Histórica en 3 Paneles...")

# 1. Extraemos los subconjuntos usando la frontera de forward test
df_hist = df_master.iloc[:idx_fwd].copy()                         # Filtro histórico (evita Data Leakage)

p_real_h = df_hist['close'].values.astype(np.float64)             # Target original (Ground Truth)
senal_k_h = df_hist['Kalman_ZeroLag'].values                      # Tendencia purificada (Fase Cero)
std_h = df_hist['StdDev'].values                                  # Volatilidad histórica (Riesgo)
res_target_h = df_hist['Residual_Micro'].values                   # Ruido microestructural (Target)
fechas_h = df_hist['time'].values                                 # Eje temporal para Plotly
kt_vista = df_hist['Kalman_Gain'].values                          # Historial de ganancia Kt

# 2. Matrices para el entrenamiento de validación
scaler_X_h = StandardScaler()                                     # sklearn: Normalización de entradas (media 0, var 1)
X_norm_h = scaler_X_h.fit_transform(np.column_stack((senal_k_h, std_h))) # Normaliza inputs [Kalman, Volatilidad]

scaler_y_h = StandardScaler()                                     # sklearn: Normalización del target (media 0, var 1)
y_norm_h = scaler_y_h.fit_transform(res_target_h.reshape(-1, 1))  # Normaliza target [Residuo]

T = 24
X_tens_h, y_targ_h = [], []
for i in range(len(X_norm_h) - T):
    X_tens_h.append(X_norm_h[i:i+T])                              # Ventana de entrada de 24h
    y_targ_h.append(y_norm_h[i+T, 0])                             # Residuo objetivo en la hora T+1

# Conversión a tensores finales
X_tens_h, y_targ_h = np.array(X_tens_h), np.array(y_targ_h)       # Dimensiones: X (Batch, 24, 2) | y (Batch, 1)

# Train/Test Split (Validación interna del historial)
limite_train_h = int(len(X_tens_h) * 0.85)                        # Umbral divisorio del 85% para entrenamiento
X_train_h, X_test_h = X_tens_h[:limite_train_h], X_tens_h[limite_train_h:] # inputs split: (Train, 24, 2) | (Test, 24, 2)
y_train_h, y_test_h = y_targ_h[:limite_train_h], y_targ_h[limite_train_h:] # targets split: (Train, 1) | (Test, 1)

# 3. Construcción del cerebro de auditoría (Arquitectura)
entrada_h = Input(shape=(T, 2))                                   # TF: Entrada simbólica. Dimensión: (24, 2)
x_h = LSTM(64, return_sequences=True, activation='tanh')(entrada_h) # TF: LSTM 1. Unidades: 64. Salida: (24, 64) [Secuencial]
x_h = Dropout(0.2)(x_h)                                           # TF: Regularizador. Apaga 20% de neuronas
x_h = LSTM(32, activation='tanh')(x_h)                            # TF: LSTM 2. Unidades: 32. Salida: (32) [Vector plano]
x_h = Dropout(0.2)(x_h)                                           # TF: Regularizador. Apaga 20% de neuronas
salida_res_h = Dense(1, name='Residual')(x_h)                     # TF: Capa de Salida Densa. Activación: Lineal. Dimensión: (1)

# Definición y Compilación del Grafo
model_audit = Model(inputs=entrada_h, outputs=[salida_res_h])     # TF: Ensamblaje de la red funcional
model_audit.compile(optimizer=Adam(0.001), loss='mse')            # TF: Compilación. Optimizador: Adam | Pérdida: MSE
early_stop_audit = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True) # TF: Callback. Evita overfitting

print("\n=== ARQUITECTURA DEL MODELO DE AUDITORÍA ===")
model_audit.summary()                                             # TF: Inspección de formas y parámetros de la red
print("=============================================\n")

-> Generando Auditoría Cuantitativa e Histórica en 3 Paneles...

=== ARQUITECTURA DEL MODELO DE AUDITORÍA ===


E0000 00:00:1781843887.003557   35927 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 2)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 24, 64)         │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Residual (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,601 (115.63 KB)

 Trainable params: 29,601 (115.63 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# =====================================================================
# 4. PREDICCIÓN Y RECONSTRUCCIÓN DEL PRECIO ABSOLUTO
# =====================================================================

# Train
print("Entrenando cerebro de auditoría...")
# TF: Entrenamiento de la red con validación interna del 15%
model_audit.fit(X_train_h, y_train_h, validation_split=0.15, epochs=60, batch_size=32, callbacks=[early_stop_audit], verbose=0)

# Predict
# TF: Predicción de residuos normalizados sobre el test oculto
pred_t_norm = model_audit.predict(X_test_h, verbose=0)
# sklearn: Desnormalización de predicciones a escala JPY
pred_t_res = scaler_y_h.inverse_transform(pred_t_norm).flatten()

# Alineación matemática de vectores temporales
kalman_test_base = senal_k_h[T + limite_train_h :]                      # Base de Kalman para test
pred_test_real = kalman_test_base + pred_t_res                            # Reconstrucción: Kalman + Residuo predicho
reales_test = p_real_h[T + limite_train_h :]                            # Precios reales de control (Test)
fechas_test = fechas_h[T + limite_train_h :]                            # Fechas correspondientes al test

# =====================================================================
# 5. RENDERIZADO DEL DASHBOARD DE 3 PANELES (CON GANANCIA KT RESTAURADA)
# =====================================================================
vista = -1000                                                           # Ventana de visualización (últimas 1000h)
f_vista = fechas_h[vista:]                                              # Fechas históricas de control
p_real_vista = p_real_h[vista:]                                         # Precios históricos de control
k_senal_vista = senal_k_h[vista:]                                       # Señal Kalman histórica de control

# Plotly: Inicialización del dashboard de 3 niveles
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        '1. Acción del Precio vs Filtro Kalman Reactivo (Cero Desfase)',
        '2. Dinámica de la Ganancia Kalman (K_t Frontal)',
        '3. Auditoría de Red LSTM: Aprendizaje Residual vs Precio Real'
    )
)

# PANEL 1: Precio Real vs Kalman Reactivo
fig.add_trace(go.Scatter(x=f_vista, y=p_real_vista, mode='lines', name='Precio Real', line=dict(color='gray', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=f_vista, y=k_senal_vista, mode='lines', name='Kalman Reactivo', line=dict(color='orange', width=2)), row=1, col=1)

# PANEL 2: Dinámica de la Ganancia Kalman (Kt)
fig.add_trace(go.Scatter(x=f_vista, y=kt_vista[vista:], mode='lines', name='Ganancia Kt', line=dict(color='teal', width=1.5)), row=2, col=1)
fig.add_hline(y=0.70, line_dash="dash", line_color="red", row=2, col=1) # Límite superior de control

# PANEL 3: Comparativa de Validación LSTM (Test Oculto)
f_test_v = fechas_test[vista:] if len(fechas_test) > abs(vista) else fechas_test
r_test_v = reales_test[vista:] if len(reales_test) > abs(vista) else reales_test
p_test_v = pred_test_real[vista:] if len(pred_test_real) > abs(vista) else pred_test_real

fig.add_trace(go.Scatter(x=f_test_v, y=r_test_v, mode='lines', name='Real (Test)', line=dict(color='black', width=2.5)), row=3, col=1)
fig.add_trace(go.Scatter(x=f_test_v, y=p_test_v, mode='lines', name='LSTM Residual (Test)', line=dict(color='red', width=2)), row=3, col=1)

# Cálculo de la métrica de error promedio (MAPE)
mape_test = np.nanmean(np.abs((r_test_v - p_test_v) / r_test_v)) * 100
fig.update_layout(height=950, hovermode='x unified', template='plotly_white',
                  title_text=f"Kalman Reactivo + LSTM Residual | {symb} | MAPE Test: {mape_test:.4f}%")

# Plotly: Oclusión de fines de semana para evitar vacíos visuales
fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])], row=1, col=1)
fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])], row=2, col=1)
fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])], row=3, col=1)

fig.show()

Entrenando cerebro de auditoría...


In [4]:
# =====================================================================
# 3. PIPELINE DE PRODUCCIÓN Y FORWARD TESTING
# =====================================================================
print("-> Entrenando Modelo Final de Producción (100% Data Histórica)...")

# ---------------------------------------------------------------------
# FASE 1: EXTRACCIÓN DE DATOS DEL MASTER DATASET
# ---------------------------------------------------------------------
# Extracción de vectores del Master Dataset
p_real_c = df_master['close'].values.astype(np.float64)             # Target original bruto (JPY)
senal_kalman_c = df_master['Kalman_ZeroLag'].values                # Señal limpia (Fase Cero)
std_c = df_master['StdDev'].values                                  # Volatilidad móvil de 14 periodos
residual_c = df_master['Residual_Micro'].values                    # Ruido microestructural de control
fechas_c = df_master['time'].values                                 # Eje cronológico para Plotly

# ---------------------------------------------------------------------
# FASE 2: NORMALIZACIÓN Y ESCALADO INDEPENDIENTE
# ---------------------------------------------------------------------
# sklearn: Escalado para las entradas [Kalman, Volatilidad] (media 0, var 1)
scaler_X_c = StandardScaler()
X_norm_c = scaler_X_c.fit_transform(np.column_stack((senal_kalman_c, std_c)))

# sklearn: Escalado para el target de residuo (media 0, var 1)
scaler_y_res_c = StandardScaler()
y_res_norm_c = scaler_y_res_c.fit_transform(residual_c.reshape(-1, 1))

# sklearn: Escalado para el target de riesgo (min 0, max 1)
scaler_y_std_c = MinMaxScaler()
y_std_norm_c = scaler_y_std_c.fit_transform(std_c.reshape(-1, 1))

# ---------------------------------------------------------------------
# FASE 3: CONSTRUCCIÓN DE TENSORES (VENTANAS DE TIEMPO 'T')
# ---------------------------------------------------------------------
X_all, y_res_all, y_std_all, fechas_all, k_base_all = [], [], [], [], []

# Generación de ventanas secuenciales temporales
for i in range(len(X_norm_c) - T):
    X_all.append(X_norm_c[i:i+T])              # Ventana de entrada de 24h
    y_res_all.append(y_res_norm_c[i+T, 0])     # Target 1 (Residuo) en i+T
    y_std_all.append(y_std_norm_c[i+T, 0])     # Target 2 (Riesgo) en i+T
    fechas_all.append(fechas_c[i+T])           # Sincronización del tiempo en i+T
    k_base_all.append(senal_kalman_c[i+T])     # Base de Kalman para de-escalado

# Conversión a tensores finales
X_all = np.array(X_all)                        # Shape: (Batch, 24, 2)
y_res_all, y_std_all = np.array(y_res_all), np.array(y_std_all) # Shapes: (Batch, 1) | (Batch, 1)
fechas_all = pd.to_datetime(fechas_all)        # Eje de tiempo de los tensores
k_base_all = np.array(k_base_all)              # Base de Kalman para test

# ---------------------------------------------------------------------
# FASE 4: DEFINICIÓN DE LA FRONTERA TEMPORAL (EVITAR DATA LEAKAGE)
# ---------------------------------------------------------------------
# Límite divisorio exacto para evitar data leakage
idx_fwd_arr = np.searchsorted(fechas_all, fecha_inicio_forward)

# ---------------------------------------------------------------------
# FASE 5: ARQUITECTURA DE LA RED NEURONAL (LSTM DUAL-HEAD)
# ---------------------------------------------------------------------
entrada_f = Input(shape=(T, 2))                                    # TF: Capa de entrada. Dimensión: (24, 2)

# TF: LSTM 1. Unidades: 64. Salida: (Batch, 24, 64)
x_f = LSTM(64, return_sequences=True, activation='tanh')(entrada_f)
x_f = Dropout(0.2)(x_f)                                            # TF: Regularizador. Apaga 20% de neuronas
# TF: LSTM 2. Unidades: 32. Salida: (Batch, 32)
x_f = LSTM(32, activation='tanh')(x_f)
x_f = Dropout(0.2)(x_f)                                            # TF: Regularizador. Apaga 20% de neuronas
# TF: Capa Densa (Cuello de botella). Salida: (Batch, 32)
x_f = Dense(32, activation='tanh')(x_f)

# TF: Cabeza de salida 1 (Residuo). Activación: Lineal. Dimensión: (1)
res_f = Dense(1, name='Res')(x_f)
# TF: Cabeza de salida 2 (Riesgo). Activación: ReLU (forzada a positivo). Dimensión: (1)
std_f = Dense(1, activation='relu', name='Std')(x_f)

# TF: Ensamblaje del grafo funcional multi-cabeza
model_production = Model(inputs=entrada_f, outputs=[res_f, std_f])
# TF: Compilación. Optimizador: Adam | Pérdidas: MSE | Pesos de loss: 1.0 vs 0.3
model_production.compile(optimizer=Adam(0.001), loss={'Res': 'mse', 'Std': 'mse'}, loss_weights={'Res': 1.0, 'Std': 0.3})

print("\n=== ARQUITECTURA DEL MODELO DE PRODUCCIÓN ===")
model_production.summary()                                         # TF: Inspección estructural del modelo unificado
print("=============================================\n")

-> Entrenando Modelo Final de Producción (100% Data Histórica)...

=== ARQUITECTURA DEL MODELO DE PRODUCCIÓN ===


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 24, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 24, 64)    │     17,152 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 24, 64)    │          0 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 32)        │     12,416 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      1,056 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Res (Dense)         │ (None, 1)         │         33 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Std (Dense)         │ (None, 1)         │         33 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 30,690 (119.88 KB)

 Trainable params: 30,690 (119.88 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# TF: Callback contra sobreajuste (Monitorea val_loss)
early_stop_f = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)

# ---------------------------------------------------------------------
# FASE 6: ENTRENAMIENTO DE PRODUCCIÓN
# ---------------------------------------------------------------------
# TF: Entrenamiento con datos históricos (Excluye la semana ciega)
model_production.fit(X_all[:idx_fwd_arr], {'Res': y_res_all[:idx_fwd_arr], 'Std': y_std_all[:idx_fwd_arr]},
                     validation_split=0.15, epochs=60, batch_size=32, callbacks=[early_stop_f], verbose=0)

# ---------------------------------------------------------------------
# FASE 7: PREDICCIÓN A CIEGAS (FORWARD TESTING)
# ---------------------------------------------------------------------
# TF: Inferencia fuera de muestra (Semana ciega)
p_res_f, p_std_f = model_production.predict(X_all[idx_fwd_arr:], verbose=0)

# RECONSTRUCCIÓN MATEMÁTICA:
# sklearn: Desnormalización del residuo predicho a escala JPY
prediccion_central = k_base_all[idx_fwd_arr:] + scaler_y_res_c.inverse_transform(p_res_f).flatten()
# sklearn: Desnormalización del riesgo predicho a escala JPY
pred_std = scaler_y_std_c.inverse_transform(p_std_f).flatten()

# Canal gaussiano (+2σ) de probabilidad (95%)
banda_superior = prediccion_central + (2 * pred_std)
# Canal gaussiano (-2σ) de probabilidad (95%)
banda_inferior = prediccion_central - (2 * pred_std)

fechas_forward = fechas_all[idx_fwd_arr:]                               # Fechas del forward test
# Segmentación de datos reales de la semana ciega
df_velas_forward = df_master[df_master['time'].isin(fechas_forward)].copy()

# ---------------------------------------------------------------------
# FASE 8: RENDERIZADO DEL DASHBOARD FINAL
# ---------------------------------------------------------------------
# Plotly: Inicialización de 2 paneles temporales
fig = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.1, row_heights=[0.3, 0.7],
                    subplot_titles=('1. Contexto Histórico', '2. FORWARD TESTING (Semana Ciega)'))

# PANEL 1: Precio histórico de control
vista_k = -1000
fig.add_trace(go.Scatter(x=fechas_c[vista_k:], y=p_real_c[vista_k:], mode='lines', name='Real', line=dict(color='gray')), row=1, col=1)
# PANEL 1: Filtro de Kalman histórico de control
fig.add_trace(go.Scatter(x=fechas_c[vista_k:], y=senal_kalman_c[vista_k:], mode='lines', name='Kalman', line=dict(color='orange')), row=1, col=1)

# PANEL 2: Canal probabilístico unificado (±2σ)
fig.add_trace(go.Scatter(x=pd.concat([pd.Series(fechas_forward), pd.Series(fechas_forward)[::-1]]),
                         y=np.concatenate([banda_superior, banda_inferior[::-1]]), fill='toself', fillcolor='rgba(41,128,185,0.15)',
                         line=dict(color='rgba(255,255,255,0)'), name='Canal Predictivo (±2σ)'), row=2, col=1)
# PANEL 2: Velas japonesas de mercado real
fig.add_trace(go.Candlestick(x=df_velas_forward['time'], open=df_velas_forward['open'], high=df_velas_forward['high'],
                             low=df_velas_forward['low'], close=df_velas_forward['close'], name='Mercado'), row=2, col=1)
# PANEL 2: Línea central de consenso LSTM
fig.add_trace(go.Scatter(x=fechas_forward, y=prediccion_central, mode='lines', name='Consenso', line=dict(color='darkblue', width=3)), row=2, col=1)

# Cálculo de la métrica de error (MAPE) fuera de muestra
mape = np.mean(np.abs((df_velas_forward['close'].values - prediccion_central) / df_velas_forward['close'].values)) * 100

fig.update_layout(title=f'Producción Cuantitativa Terminada | MAPE Forward: {mape:.4f}%', height=900, template='plotly_white', xaxis2_rangeslider_visible=False)
# Plotly: Oclusión de fines de semana para evitar vacíos
fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])]) 
fig.show()

# ---------------------------------------------------------------------
# FASE 9: EXPORTACIÓN PARA TRADING EN VIVO
# ---------------------------------------------------------------------
# TF: Guardado de red neural (.keras)
model_production.save(f"modelo_dual_{symb}.keras")
# Guardado de escalador X (.pkl)
joblib.dump(scaler_X_c, f"scaler_X_dual_{symb}.pkl")
# Guardado de escalador del residuo (.pkl)
joblib.dump(scaler_y_res_c, f"scaler_y_res_dual_{symb}.pkl")
# Guardado de escalador del riesgo (.pkl)
joblib.dump(scaler_y_std_c, f"scaler_y_std_dual_{symb}.pkl")
print("[!] CEREBRO Y ESCALADORES GUARDADOS [!]")

# =====================================================================
# FASE 10: EXPORTACIÓN PARA EL BOT DE TRADING
# =====================================================================
# Duplicación de estructura para empaquetado final
df_forward_predictions = df_velas_forward.copy()

# Consolidación de predicciones matemáticas
df_forward_predictions['prediccion_central'] = prediccion_central
df_forward_predictions['banda_superior'] = banda_superior
df_forward_predictions['banda_inferior'] = banda_inferior

# Exportación física del Forecast Sheet
archivo_predictivo = f"{symb}_H1_Predictions_Blind_Week.csv"
df_forward_predictions.to_csv(archivo_predictivo, index=False)

print(f"### Pronósticos de la semana ciega guardados correctamente como: {archivo_predictivo} ###")

[!] CEREBRO Y ESCALADORES GUARDADOS [!]
### Pronósticos de la semana ciega guardados correctamente como: EURJPY_H1_Predictions_Blind_Week.csv ###
